
# 11 — Data mapping and normalization (Cars 4 You)

**Scope:** Perform Mapping and normalization (no modeling).  
Outputs a clean dataset and a JSON processing report for traceability.

ONLY Rulebased Cleaning (no ML), Missing-Handling, Encoding-Preparation.  
No Fit, no Scalers, no Target.
We want to avoid any data leakage.

That is the reason, why we don't have to split the data into train/test here.

# Table of Contents

Take this as an example for a Table of Contents for your notebook.
We have to fix all the names and sections according to what we actually do in the notebook.

<a class="anchor" id="top"></a>

** **

1. [Importing Libraries](##1.-Importing-Libraries) <br>
    
2. [Data Access & Loading](#2.-Data-Access-&-Loading) <br>
    
3. [Type Casting](#3.-Type-Casting) <br>

4. [Duplicate Removal](#3.1-Duplicate-Removal) <br>
    
5. [Category Normalization](#3.2-Category-Normalization) <br>
   
   &emsp; 3.2.1 [Data Type Conversions](#3.2.1-Data-Type-Conversions) <br>
   
   &emsp; 3.2.2 [Encoding](#3.2.2-Encoding) <br>
   
   &emsp; 3.2.3 [Other Transformations](#3.2.3-Other-Transformations) <br>
    
   &emsp; 3.2.4 [Unique Feature-Pair Analysis](#3.2.4-Unique-Feature-Pair-Analysis) <br> 

   3.3 [Train-Test Split](#3.3-Train-Test-Split) <br>
   
   3.4 [Missing Values](#3.4-Missing-Values) <br>
    
   3.5 [Outliers](#3.5-Outliers) <br>

   3.6 [Visualisations](#3.6-Visualisations) <br><br>
   

## 1. Importing Libraries

In [1]:
import os, re, math, warnings
from pathlib import Path
from datetime import datetime
import json
import pandas as pd
import numpy as np
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("mode.copy_on_write", True)
warnings.filterwarnings("ignore")

RANDOM_STATE = 42  # for reproducibility of any sampling


## 2. Data Access & Loading

Loading the data from CSV files into pandas DataFrames. 

In [2]:
# Load the data paths
data_dir = "../data/"

# Load the raw data into a pandas dataframe
train = pd.read_csv(os.path.join(data_dir, "train.csv"))
test = pd.read_csv(os.path.join(data_dir, "test.csv"))

print("Loaded shape:", train.shape)
display(train.head(3))


Loaded shape: (75973, 14)


,carID,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,69512,VW,Golf,2016.0,22290,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4.0,0.0
1,53000,Toyota,Yaris,2019.0,13790,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1.0,0.0
2,6366,Audi,Q2,2019.0,24990,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,56.0,4.0,0.0


In [3]:
# Display the NaN values for each column in a table for train set
nan_summary = train.isna().sum().reset_index()
nan_summary.columns = ['Column', 'NaN Count']
nan_summary = nan_summary[nan_summary['NaN Count'] > 0]
nan_summary = nan_summary.sort_values(by='NaN Count', ascending=False)
nan_summary

,Column,NaN Count
9,mpg,7926
8,tax,7904
12,previousOwners,1550
13,hasDamage,1548
11,paintQuality%,1524
5,transmission,1522
1,Brand,1521
2,model,1517
10,engineSize,1516
7,fuelType,1511


## 3. Type Casting

We convert the data types of the features to appropriate types.

In [4]:
def to_int_series(s):
    return pd.to_numeric(s, errors="coerce").round().astype("Int64")

def to_float_series(s):
    return pd.to_numeric(s, errors="coerce").astype(float)

nan_report = {}

num_cols_suggest = ["price","mileage","engineSize","mpg","tax","year","previousOwners"]
for col in num_cols_suggest:
    if col in train.columns:
        before = train[col].isna().sum()
        if col in ["year", "previousOwners"]:
            train[col] = to_int_series(train[col])
        else:
            train[col] = to_float_series(train[col])
        after = train[col].isna().sum()
        added = after - before
        if added > 0:
            nan_report[col] = added

# object → string cleanup
for c in train.select_dtypes(include="object").columns:
    before = train[c].isna().sum()
    train[c] = (
        train[c].astype("string").str.strip()
        .replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})
    )
    after = train[c].isna().sum()
    added = after - before
    if added > 0:
        nan_report[c] = added

# print summary
if nan_report:
    print("Columns where casting/cleaning created new NaNs:")
    for col, n in nan_report.items():
        print(f"  - {col}: +{n} NaNs")
else:
    print("No new NaNs created by casting/cleaning.")

train.head(2)


No new NaNs created by casting/cleaning.


,carID,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,69512,VW,Golf,2016,22290.0,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4,0.0
1,53000,Toyota,Yaris,2019,13790.0,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1,0.0


In [5]:
# Types of the columns
train.dtypes

carID                      int64
Brand             string[python]
model             string[python]
year                       Int64
price                    float64
transmission      string[python]
mileage                  float64
fuelType          string[python]
tax                      float64
mpg                      float64
engineSize               float64
paintQuality%            float64
previousOwners             Int64
hasDamage                float64
dtype: object

## 4. Duplicate Removal

We check for duplicates in carID and remove them if any exist.

In [6]:
# --- ensure carID is a column ---
if "carID" not in train.columns and train.index.name == "carID":
    train = train.reset_index()

# --- check for duplicate carIDs (do NOT drop) ---
dup_mask = train.duplicated(subset=["carID"], keep=False)
dup_rows = train.loc[dup_mask].sort_values("carID")

n_dups = train.duplicated(subset=["carID"]).sum()
print(f"[TRAIN] duplicate carID count: {n_dups}")

if n_dups > 0:
    print("carID values with duplicates:")
    print(dup_rows["carID"].unique())
    print("\nRows with duplicated carID:")
    display(dup_rows)
else:
    print("No duplicate carID found.")


[TRAIN] duplicate carID count: 0
No duplicate carID found.


In [7]:
# --- ensure carID is a column ---
if "carID" not in test.columns and test.index.name == "carID":
    test = test.reset_index()

# --- check for duplicate carIDs (do NOT drop) ---
dup_mask_test = test.duplicated(subset=["carID"], keep=False)
dup_rows_test = test.loc[dup_mask_test].sort_values("carID")

n_dups_test = test.duplicated(subset=["carID"]).sum()
print(f"[TEST] duplicate carID count: {n_dups_test}")

if n_dups_test > 0:
    print("carID values with duplicates:")
    print(dup_rows_test["carID"].unique())
    print("\nRows with duplicated carID:")
    display(dup_rows_test)
else:
    print("No duplicate carID found in test.")


[TEST] duplicate carID count: 0
No duplicate carID found in test.


Since carID is a unique identifier, we can use it as the index of the DataFrame for easier data manipulation.

In [8]:
# Set carID as index for training and test dataframes
train.set_index('carID', inplace=True)
test.set_index('carID', inplace=True)

## 5. Category Normalization and Mapping for the train and test dataset

In [9]:
# We make all string columns to lowercase for consistency
for c in train.select_dtypes(include="string").columns:
    train[c] = train[c].str.lower()
for c in test.select_dtypes(include="string").columns:
    test[c] = test[c].str.lower()

## 5.1 Transmission Normalization and Mapping


In [10]:
def normalize_transmission(value: str):
    if pd.isna(value):
        return pd.NA
    value = str(value).strip().lower()
    value = re.sub(r"[.,_]", " ", value)
    value = " ".join(value.split())
    return value

def apply_transmission_mapping(df: pd.DataFrame,
                               mapping_path: str = "../mapping/transmission_mapping.json",
                               col: str = "transmission") -> pd.DataFrame:
    # load + normalize mapping keys
    with open(mapping_path, "r", encoding="utf-8") as f:
        raw_map = json.load(f)

    trans_canon = {normalize_transmission(k): v for k, v in raw_map.items()}


    # debug before
    uniq_before = df[col].nunique(dropna=True)
    print(f"[DEBUG] {col}: unique values BEFORE mapping: {uniq_before}")


    # normalize column
    norm_col = f"{col}_norm"
    df[norm_col] = df[col].apply(normalize_transmission)

    # map
    df[col] = df[norm_col].map(trans_canon)

    # debug after
    uniq_after = df[col].nunique(dropna=True)
    print(f"[DEBUG] {col}: unique values AFTER mapping: {uniq_after}")
    print(f"[DEBUG] {col}: sample AFTER:", df[col].dropna().unique()[:15])

    # check unmapped
    unmapped = (
        df.loc[df[col].isna(), norm_col]
          .dropna()
          .unique()
    )
    if len(unmapped) > 0:
        print(f"[WARN] Unmapped {col} values:")
        for v in unmapped:
            print("  -", repr(v))
    else:
        print(f"[INFO] All {col} values were mapped.")

    # drop helper
    df.drop(columns=[norm_col], inplace=True)

    return df

# use it for both
train = apply_transmission_mapping(train)
test  = apply_transmission_mapping(test)


[DEBUG] transmission: unique values BEFORE mapping: 17
[DEBUG] transmission: unique values AFTER mapping: 5
[DEBUG] transmission: sample AFTER: ['Semi-Auto' 'Manual' 'Automatic' 'Unknown' 'Other']
[INFO] All transmission values were mapped.
[DEBUG] transmission: unique values BEFORE mapping: 38
[DEBUG] transmission: unique values AFTER mapping: 5
[DEBUG] transmission: sample AFTER: ['Automatic' 'Semi-Auto' 'Manual' 'Unknown' 'Other']
[INFO] All transmission values were mapped.


## 5.2 Fuel Type Normalization and Mapping

In [11]:
def normalize_fuel(value: str):
    """Normalize raw fuelType text before mapping."""
    if pd.isna(value):
        return pd.NA
    value = str(value).strip().lower()
    value = re.sub(r"[.,-_]", " ", value)  # unify separators
    value = " ".join(value.split())        # collapse spaces
    return value

def apply_fuel_mapping(
    df: pd.DataFrame,
    mapping_path: str = "../mapping/fueltype_mapping.json",
    col: str = "fuelType"
) -> pd.DataFrame:
    # load + normalize mapping keys
    with open(mapping_path, "r", encoding="utf-8") as f:
        raw_fuel_map = json.load(f)

    FUEL_CANON = {normalize_fuel(k): v for k, v in raw_fuel_map.items()}

    # debug before only count unique values
    uniq_before = df[col].nunique(dropna=True)
    print(f"[DEBUG] {col}: unique values BEFORE mapping: {uniq_before}")

    norm_col = f"{col}_norm"
    df[norm_col] = df[col].apply(normalize_fuel)

    # map
    df[col] = df[norm_col].map(FUEL_CANON)

    # debug after
    print(f"[DEBUG] {col}: unique AFTER:", df[col].dropna().unique()[:20])

    # report unmapped
    unmapped = (
        df.loc[df[col].isna(), norm_col]
          .dropna()
          .unique()
    )
    if len(unmapped) > 0:
        print(f"[WARN] Unmapped {col} values (add to JSON):")
        for v in unmapped:
            print("  -", repr(v))
    else:
        print(f"[INFO] All {col} values were mapped.")

    # drop helper
    df.drop(columns=[norm_col], inplace=True)

    return df

# usage
train = apply_fuel_mapping(train)
test  = apply_fuel_mapping(test)


[DEBUG] fuelType: unique values BEFORE mapping: 16
[DEBUG] fuelType: unique AFTER: ['Petrol' 'Diesel' 'Hybrid' 'Other' 'Electric']
[INFO] All fuelType values were mapped.
[DEBUG] fuelType: unique values BEFORE mapping: 29
[DEBUG] fuelType: unique AFTER: ['Petrol' 'Diesel' 'Hybrid' 'Other' 'Electric']
[INFO] All fuelType values were mapped.


## 5.3 Normalizing Brand Names with Mapping 

We have typos in our brand names. We create a mapping dictionary to correct them. 

In [12]:
def normalize_brand(value: str):
    if pd.isna(value):
        return pd.NA
    value = str(value).strip().lower()
    value = re.sub(r"[.,-_]", " ", value)
    value = " ".join(value.split())
    return value

def apply_brand_mapping(
    df: pd.DataFrame,
    mapping_path: str = "../mapping/brandname_mapping.json",
    col: str = "Brand"
) -> pd.DataFrame:
    if col not in df.columns:
        print(f"[INFO] Column '{col}' not found.")
        return df

    # load mapping
    with open(mapping_path, "r", encoding="utf-8") as f:
        raw_brand_map = json.load(f)

    # normalize mapping keys
    BRAND_CANON = {normalize_brand(k): v for k, v in raw_brand_map.items()}

    print(f"\n[INFO] {col} BEFORE cleaning:")
    print(df[col].dropna().unique()[:30])

    was_nan_before = df[col].isna()

    # normalize column
    norm_col = f"{col}_norm"
    df[norm_col] = df[col].apply(normalize_brand)

    # map
    mapped = df[norm_col].map(BRAND_CANON)

    # keep original where no mapping exists
    clean_col = f"{col}_clean"
    df[clean_col] = df[col].where(mapped.isna(), mapped)

    # detect new NaNs that came from missing mapping
    new_nans_mask = df[clean_col].isna() & (~was_nan_before)
    new_nans_count = new_nans_mask.sum()

    # collect unmapped originals
    unmapped = (
        df.loc[mapped.isna() & (~was_nan_before) & df[norm_col].notna(), norm_col]
          .dropna()
          .unique()
    )

    # replace original
    df[col] = df[clean_col]
    df.drop(columns=[norm_col, clean_col], inplace=True)

    print(f"\n[INFO] {col} AFTER cleaning:")
    print(df[col].dropna().unique()[:30])

    if new_nans_count > 0:
        print(f"[WARN] {new_nans_count} rows became NaN due to missing mapping.")
    if len(unmapped) > 0:
        print("\n[WARN] Unmapped brand values (add to JSON):")
        for v in unmapped:
            print("  -", repr(v))

    return df

# usage
train = apply_brand_mapping(train, "../mapping/brandname_mapping.json", col="Brand")
test  = apply_brand_mapping(test,  "../mapping/brandname_mapping.json", col="Brand")



[INFO] Brand BEFORE cleaning:
<StringArray>
[      'vw',   'toyota',     'audi',     'ford',      'bmw',    'skoda',     'opel', 'mercedes',      'for',  'hyundai',        'w',      'ord',       'mw',
   'yundai',       'bm',    'toyot',      'udi',      'ope',        'v',      'pel',       'pe',  'mercede',     'koda',   'hyunda',      'aud',  'ercedes',
    'oyota',     'skod',      'kod',    'yunda']
Length: 30, dtype: string

[INFO] Brand AFTER cleaning:
<StringArray>
['Volkswagen', 'Toyota', 'Audi', 'Ford', 'BMW', 'Škoda', 'Opel', 'Mercedes-Benz', 'Hyundai']
Length: 9, dtype: string

[INFO] Brand BEFORE cleaning:
['Hyundai' 'VW' 'BMW' 'Opel' 'Ford' 'Mercedes' 'Skoda' 'Toyot' 'Toyota'
 'Audi' 'For' 'Ope' 'toyota' 'vw' 'hyundai' 'MW' 'SKODA' 'ord' 'udi' 'bmw'
 'V' 'BM' 'HYUNDAI' 'OPEL' 'mercedes' 'audi' 'Mercede' 'pel' 'opel' 'FORD']

[INFO] Brand AFTER cleaning:
['Hyundai' 'Volkswagen' 'BMW' 'Opel' 'Ford' 'Mercedes-Benz' 'Škoda'
 'Toyota' 'Audi']


## 5.4 Normalizing Model Names with Mapping

Like we did for brand names, we also have typos in model names. We create a mapping dictionary to correct them.
The ones, we couldnt map, we will make them na, so we can impute them later.

In [13]:
# Look into the unique model names
unique_models = train["model"].dropna().unique().tolist()
print("\nUnique Model in 'model':\n", unique_models)
print(f"\nNumber of unique models: {len(unique_models)}")



Unique Model in 'model':
 ['golf', 'yaris', 'q2', 'fiesta', '2 series', '3 series', 'a3', 'octavia', 'passat', 'focus', 'insignia', 'a clas', 'q3', 'fabia', 'a class', 'ka+', 'glc class', 'i30', 'c class', 'polo', 'e class', 'c clas', 'q5', 'up', 'c-hr', 'mokka x', 'corsa', 'astra', 'tt', '5 series', 'aygo', '4 series', 'slk', 'viva', 't-roc', 'ecosport', 'tucson', 'ecospor', 'x-class', 'cl class', 'ix20', 'i20', 'rapid', 'a1', 'auris', 'sharan', 'adam', 'x3', 'a8', 'gls class', 'b-max', 'a4', 'kona', 'i10', 'mokka', 's-max', 'x2', 'crossland x', 'tiguan', 'a5', 'gle class', 'zafira', 'ioniq', 'a6', 'mondeo', 'yeti outdoor', 'x1', 'scala', 's class', '1 series', 'kamiq', 'kuga', 'tourneo connect', 'q7', 'gla class', 'arteon', 'sl class', 'santa fe', 'grandland x', 'i800', 'rav4', 'touran', 'citigo', 'focu', 'roomster', 'prius', 'corolla', 'fiest', 'b class', 'q', 'kodiaq', 'v class', 'e clas', 'caddy maxi life', 'superb', 'astr', 'getz', 'combo life', 'beetle', 'galaxy', 'm3', 'gtc', 

In [14]:
import json, re
import pandas as pd

def norm_model(s):
    if pd.isna(s):
        return pd.NA
    s = str(s).strip().lower()
    s = re.sub(r"[.,\-_ ]+", "", s)
    return s

def apply_model_mapping(
    df: pd.DataFrame,
    mapping_path: str = "../mapping/modelname_mapping.json",
    col: str = "model"
) -> pd.DataFrame:
    # load JSON
    with open(mapping_path, "r", encoding="utf-8") as f:
        cfg = json.load(f)

    raw_aliases = cfg["aliases"]
    regex_rules = cfg.get("regex_rules", [])

    # normalize alias keys
    ALIASES = {norm_model(k): v for k, v in raw_aliases.items()}

    def apply_regex_first(val: str) -> str:
        for rule in regex_rules:
            pat = rule["pattern"]
            repl = rule["replace"]
            m = re.fullmatch(pat, val)
            if m:
                return m.expand(repl)
        return val

    if col not in df.columns:
        print(f"[INFO] column '{col}' not in df, skipping.")
        return df

    # debug before
    print(f"\n[INFO] {col} BEFORE mapping: {df[col].nunique(dropna=True)} unique")

    # normalize incoming values
    df[f"{col}_norm"] = df[col].apply(norm_model)

    # map
    def map_model(norm_val):
        if pd.isna(norm_val):
            return pd.NA
        norm_val = apply_regex_first(norm_val)
        return ALIASES.get(norm_val, pd.NA)

    df[f"{col}_mapped"] = df[f"{col}_norm"].apply(map_model)

    # overwrite with mapped
    df[col] = df[f"{col}_mapped"]

    # everything that had a value (norm not NA) but no mapping -> set to NA
    mask_unmapped = df[col].isna() & df[f"{col}_norm"].notna()
    df.loc[mask_unmapped, col] = pd.NA

    # clean up
    df.drop(columns=[f"{col}_norm", f"{col}_mapped"], inplace=True)

    # debug after
    print(f"[INFO] {col} AFTER mapping: {df[col].nunique(dropna=True)} unique")
    return df

# use for both
train = apply_model_mapping(train, "../mapping/modelname_mapping.json", col="model")
test  = apply_model_mapping(test,  "../mapping/modelname_mapping.json", col="model")



[INFO] model BEFORE mapping: 296 unique
[INFO] model AFTER mapping: 185 unique

[INFO] model BEFORE mapping: 593 unique
[INFO] model AFTER mapping: 177 unique


## 5.5 Mapping missing Brand values, based on Model names

When we know a model name, we can also infer the brand name. We create a mapping dictionary for this as well.

In [15]:
def fill_brand_from_model(
    df: pd.DataFrame,
    mapping_path: str = "../mapping/brand_model_mapping.json"
) -> pd.DataFrame:
    """
    Fills df['Brand'] when:
      - Brand is NaN
      - model has a value
      - model exists in the JSON (brand_model_mapping.json)
    """
    # load JSON: {"A1": "Audi", "Golf": "Volkswagen", ...}
    with open(mapping_path, "r", encoding="utf-8") as f:
        model_to_brand = json.load(f)

    # debug before
    before = df["Brand"].isna().sum()
    print(f"[DEBUG] Brand NaN before: {before}")

    # rows where Brand is missing but model is present
    mask = df["Brand"].isna() & df["model"].notna()

    # map model -> Brand
    mapped = df.loc[mask, "model"].map(model_to_brand)

    # write back only where we actually found a brand
    fill_mask = mask.copy()
    fill_mask.loc[mask] = mapped.notna()

    df.loc[fill_mask, "Brand"] = mapped[mapped.notna()]

    # debug after
    after = df["Brand"].isna().sum()
    print(f"[DEBUG] Brand NaN after: {after}")
    print("[DEBUG] Filled examples (model -> Brand):")
    print(
        df.loc[fill_mask, ["model", "Brand"]]
          .head(4)
          .to_string(index=False)
    )

    return df

# call of the function
train = fill_brand_from_model(train, "../mapping/brand_model_mapping.json")
test = fill_brand_from_model(test, "../mapping/brand_model_mapping.json")


[DEBUG] Brand NaN before: 1521
[DEBUG] Brand NaN after: 116
[DEBUG] Filled examples (model -> Brand):
model      Brand
T-Roc Volkswagen
   A3       Audi
  i20    Hyundai
 Polo Volkswagen
[DEBUG] Brand NaN before: 649
[DEBUG] Brand NaN after: 42
[DEBUG] Filled examples (model -> Brand):
  model         Brand
Mokka X          Opel
A-Class Mercedes-Benz
 Tucson       Hyundai
    i20       Hyundai


## 6. Fix outliers 

We have a lot of outliers in our dataset. We will fix them based on domain knowledge and statistical methods.

## Year before 2000

In [16]:
# Show year values before 2000

print("Year values before 2000:")
print(train.loc[train['year'] < 2000, 'year'].unique())

Year values before 2000:
<IntegerArray>
[1996, 1998, 1970, 1999, 1997]
Length: 5, dtype: Int64


We only have 5 entries for year before 2000. We delete these rows from the dataset as they are not significant enough to keep and might skew our analysis.

In [17]:
# Drop the entries with year before 2000
train = train[train['year'] >= 2000]
test = test[test['year'] >= 2000]

## Engine Size below 1.0L and above 6.2L

We have seen in data exploration that there are some outliers in engine size below 1.0L and above 6.2L.
From our domain knowledge, we know that cars in our dataset should have engine sizes between 1.0L and 6.2L.
We will put these values to na, so we can impute them later.

In [18]:
# Count affected entries
low_engine_size_count = train[train['engineSize'] < 1.0].shape[0]
high_engine_size_count = train[train['engineSize'] > 6.2].shape[0]
print(f"Number of entries with engine size below 1.0L:  {low_engine_size_count}")
print(f"Number of entries with engine size above 6.2L:  {high_engine_size_count}")

# the values just look wrong, so we will set all unusual values to na
train.loc[((train['engineSize'] > 6.2) | (train['engineSize'] < 0.8)), 'engineSize'] = np.nan
test.loc[((test['engineSize'] > 6.2) | (test['engineSize'] < 0.8)), 'engineSize'] = np.nan

Number of entries with engine size below 1.0L:  562
Number of entries with engine size above 6.2L:  1


## 7. Process errors found in data exploration

In [19]:
# we cannot just round them, because they wouldnt match the other values, so we will set them na
train.loc[train['year'] % 1 != 0, 'year'] = np.nan
test.loc[test['year'] % 1 != 0, 'year'] = np.nan
# its obviously no sign error, so the values will be set to na for now
train.loc[train['year'] < 0, 'mileage'] = np.nan
test.loc[test['year'] < 0, 'mileage'] = np.nan

In [20]:
# The negative values can't really be explained, so we will set them to na
train.loc[train['tax'] < 0, 'tax'] = np.nan
test.loc[test['tax'] < 0, 'tax'] = np.nan

In [21]:
# its obviously no sign error, so the values will be set to na for now
train.loc[train['mileage'] < 0, 'mileage'] = np.nan
test.loc[test['mileage'] < 0, 'mileage'] = np.nan

In [22]:
# Some mpg values could be unrealistic as well
unusual_mpg_mask = ((train['mpg'] > 150) & (train["fuelType"] == "Electric")) | \
       ((train['mpg'] < 70) & (train["fuelType"] == "Electric")) | \
       ((train['mpg'] > 100) & (train["fuelType"] == "Hybrid")) | \
       ((train['mpg'] < 35) & (train["fuelType"] == "Hybrid")) | \
       ((train['mpg'] > 80) & (train["fuelType"] != "Hybrid") & (train["fuelType"] != "Electric")) | \
       ((train['mpg'] < 8) & (train["fuelType"] != "Hybrid")& (train["fuelType"] != "Electric"))
train.loc[unusual_mpg_mask, 'mpg'] = np.nan

# Same for test set
unusual_mpg_mask_test = ((test['mpg'] > 150) & (test["fuelType"] == "Electric")) | \
       ((test['mpg'] < 70) & (test["fuelType"] == "Electric")) | \
       ((test['mpg'] > 100) & (test["fuelType"] == "Hybrid")) | \
       ((test['mpg'] < 35) & (test["fuelType"] == "Hybrid")) | \
       ((test['mpg'] > 80) & (test["fuelType"] != "Hybrid") & (test["fuelType"] != "Electric")) | \
       ((test['mpg'] < 8) & (test["fuelType"] != "Hybrid")& (test["fuelType"] != "Electric"))
test.loc[unusual_mpg_mask_test, 'mpg'] = np.nan

In [23]:
# its obviously no sign error, so the values will be set to na for now
train.loc[train['previousOwners'] < 0, 'previousOwners'] = np.nan

## 10. Save Processed Data

In [24]:
# save the processed dataframe to data/processed_data for train and test
PROCESSED_CSV = os.path.join(data_dir, "processed_data/11_processed_train_data.csv")
print("Saving processed file to:", PROCESSED_CSV)
train.to_csv(PROCESSED_CSV, index=False)

# Save the test set
PROCESSED_CSV_TEST = os.path.join(data_dir, "processed_data/11_processed_test_data.csv")
print("Saving processed file to:", PROCESSED_CSV_TEST)
test.to_csv(PROCESSED_CSV_TEST, index=False)



Saving processed file to: ../data/processed_data/11_processed_train_data.csv
Saving processed file to: ../data/processed_data/11_processed_test_data.csv


In [25]:

# Display the NaN values for each column in a table for train set
nan_summary = train.isna().sum().reset_index()
nan_summary.columns = ['Column', 'NaN Count']
nan_summary = nan_summary[nan_summary['NaN Count'] > 0]
nan_summary = nan_summary.sort_values(by='NaN Count', ascending=False)
nan_summary

,Column,NaN Count
8,mpg,9004
7,tax,8108
9,engineSize,2055
11,previousOwners,1888
5,mileage,1801
1,model,1653
12,hasDamage,1521
4,transmission,1494
10,paintQuality%,1492
6,fuelType,1479


In [26]:
# Display the NaN values for each column in a table for test set
nan_summary_test = test.isna().sum().reset_index()
nan_summary_test.columns = ['Column', 'NaN Count']
nan_summary_test = nan_summary_test[nan_summary_test['NaN Count'] > 0]
nan_summary_test = nan_summary_test.sort_values(by='NaN Count', ascending=False)
nan_summary_test


,Column,NaN Count
7,mpg,3756
6,tax,3394
8,engineSize,858
4,mileage,834
1,model,713
5,fuelType,644
9,paintQuality%,617
3,transmission,614
11,hasDamage,585
10,previousOwners,583
